<a href="https://colab.research.google.com/github/mindjobs/Magister-/blob/main/UCTVirtual_Hito1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCTVirtual Advance · Hito 1 reproducible (Google Colab)
**Proyecto:** Asistente conversacional de admisión con extracción de leads, respuestas documentadas (RAG) y futura derivación a consejeros.

**Instrucciones:** Ejecutar las celdas en orden. Cargar documentos oficiales PDF/DOCX/TXT/MD y opcionalmente el ZIP borrador. No publicar datos personales reales ni claves de API en GitHub. La información institucional no se presume: toda respuesta evaluable debe tener evidencia literal.

**Entrega:** `docs/hito1.md`, `eval/cases.jsonl` (mínimo 50), `eval/annotation-rules.md`, `eval/baseline/outputs.jsonl`, `eval/baseline/metrics.json`, script reproducible y etiqueta GitHub `hito1`. Fecha según pauta: 08-11-2026 23:59 (Chile).

## 0 · Configuración y dependencias
Ejecutable en Colab o Jupyter local. No se necesita API pagada para la línea base. Las alternativas con LLM quedan como extensión opcional; **no se inventan métricas**.

In [1]:
%pip -q install pymupdf python-docx scikit-learn pandas nbformat

In [2]:
from pathlib import Path
import os, re, json, zipfile, shutil, hashlib, unicodedata, time, collections
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
ROOT=Path('/content/uctvirtual_hito1') if Path('/content').exists() else Path.cwd()/'uctvirtual_hito1'
for d in ['corpus/originales','corpus/textos','docs','eval/baseline','eval/revision']:(ROOT/d).mkdir(parents=True,exist_ok=True)
SEED_ZIP=ROOT/'borrador.zip'
print('Directorio:',ROOT)

Directorio: /content/uctvirtual_hito1


## 1 · Incorporar documentos oficiales UCTVirtual
Carga **solo documentos autorizados**: fichas de programas Advance, requisitos, aranceles, admisión, calendarios, modalidad, protección de datos, canales y protocolos de derivación. Evita fichas con datos de postulantes. Si un PDF es escaneado, se marcará como no extraíble y necesitará OCR/revisión manual.

In [3]:
# En Colab abre el selector de archivos. En local, copia archivos a corpus/originales.
try:
 from google.colab import files
 print('Selecciona documentos PDF, DOCX, TXT o MD (puedes cancelar si aún no los tienes).')
 uploaded=files.upload()
 for name, data in uploaded.items():
  if Path(name).suffix.lower() in {'.pdf','.docx','.txt','.md'}:
   (ROOT/'corpus/originales'/Path(name).name).write_bytes(data)
  elif name.lower().endswith('.zip'):
   SEED_ZIP.write_bytes(data)
except ImportError:
 print('Entorno local: copiar documentos a',ROOT/'corpus/originales')
print('Documentos:',[p.name for p in (ROOT/'corpus/originales').iterdir()])

Selecciona documentos PDF, DOCX, TXT o MD (puedes cancelar si aún no los tienes).


Saving metricas_uct_virtual.xlsx to metricas_uct_virtual.xlsx
Documentos: []


In [13]:
import fitz
from docx import Document

def extraer(p):
 if p.suffix.lower()=='.pdf':
  with fitz.open(p) as doc: return '\n'.join(f'[Página {i+1}]\n'+page.get_text() for i,page in enumerate(doc))
 if p.suffix.lower()=='.docx':
  doc=Document(p); parts=[x.text for x in doc.paragraphs]
  for table in doc.tables:
   for row in table.rows: parts.append(' | '.join(c.text for c in row.cells))
  return '\n'.join(parts)
 if p.suffix.lower() in {'.txt','.md'}:return p.read_text(encoding='utf-8-sig')
 return ''
manifest=[]
for p in sorted((ROOT/'corpus/originales').iterdir()):
 if p.suffix.lower() not in {'.pdf','.docx','.txt','.md'}:continue
 txt=extraer(p).strip()
 doc_id='uct-'+hashlib.sha256(p.name.encode()).hexdigest()[:10]
 (ROOT/'corpus/textos'/f'{doc_id}.txt').write_text(txt,encoding='utf-8')
 manifest.append({'doc_id':doc_id,'archivo':p.name,'caracteres':len(txt),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'estado':'REVISAR OCR' if len(txt)<80 else 'extraido'})
(ROOT/'corpus/manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(manifest) if manifest else pd.DataFrame(columns=['doc_id','archivo','caracteres','estado']))
if not manifest:print('SIN CORPUS OFICIAL: las preguntas con respaldo deben esperar a la carga documental.')

,doc_id,archivo,caracteres,estado


SIN CORPUS OFICIAL: las preguntas con respaldo deben esperar a la carga documental.


## 2 · Corpus trazable, fragmentación y búsqueda
Cada fragmento conserva `doc_id`, número y texto exacto. La búsqueda TF-IDF constituye la línea base RAG. Se pueden comparar tamaños de fragmento y `k` sin proveedor externo.

In [14]:
def construir_chunks(max_chars=900, overlap=120):
 chunks=[]
 for p in sorted((ROOT/'corpus/textos').glob('*.txt')):
  t=p.read_text(encoding='utf-8')
  if not t.strip():continue
  start=0; i=0
  while start<len(t):
   end=min(len(t),start+max_chars)
   if end<len(t):
    cut=t.rfind(' ',start+max_chars//2,end)
    if cut>start:end=cut
   chunk=t[start:end].strip()
   if chunk:chunks.append({'chunk_id':f'{p.stem}-c{i:03d}','doc_id':p.stem,'text':chunk,'inicio':start,'fin':end})
   if end==len(t):break
   start=max(start+1,end-overlap);i+=1
 return chunks
CHUNKS=construir_chunks()
(ROOT/'corpus/chunks.jsonl').write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in CHUNKS),encoding='utf-8')
def buscar(query,k=5,chunks=None):
 chunks=CHUNKS if chunks is None else chunks
 if not chunks:return []
 vec=TfidfVectorizer(strip_accents='unicode',ngram_range=(1,2))
 mat=vec.fit_transform([c['text'] for c in chunks]+[query])
 scores=cosine_similarity(mat[-1],mat[:-1]).ravel()
 return [dict(chunks[i],score=float(scores[i])) for i in scores.argsort()[::-1][:k] if scores[i]>0]
print('Fragmentos:',len(CHUNKS));print('Ejemplo:',CHUNKS[0] if CHUNKS else 'Pendiente de documentos')

Fragmentos: 0
Ejemplo: Pendiente de documentos


## 3 · Casos anotados: 25 extracción + 25 respuestas
Se puede importar el ZIP preliminar para reutilizar sus **casos de extracción**, pero sus preguntas y citas didácticas **NO se consideran evidencia oficial**. Se generan 25 borradores de preguntas de respuesta, pendientes de revisión humana, a partir de fragmentos reales; no se inventan respuestas. Edita `eval/cases.jsonl` antes de la evaluación definitiva.

In [15]:
SEED=[]
if SEED_ZIP.exists():
 with zipfile.ZipFile(SEED_ZIP) as z:
  if 'eval/cases.jsonl' in z.namelist():
   SEED=[json.loads(s) for s in z.read('eval/cases.jsonl').decode().splitlines() if s.strip()]
print('Casos preliminares importados:',len(SEED))
# Plantillas de extracción, sintéticas y sin datos personales reales
extract=[c for c in SEED if c.get('task')=='extract'][:25]
if not extract:
 examples=[
 ('Hola, soy Ana Pérez, correo ana@example.com, me interesa un programa Advance y autorizo que me contacten.',{'nombre':'Ana Pérez','email':'ana@example.com','telefono':None,'programa':None,'modalidad':None,'contacto_autorizado':True}),
 ('Soy Luis Soto. Mi número es +56 9 1234 5678. Quiero conocer la oferta.',{'nombre':'Luis Soto','email':None,'telefono':'56912345678','programa':None,'modalidad':None,'contacto_autorizado':None}),
 ('No quiero que me llamen. Mi correo es prueba@example.org.',{'nombre':None,'email':'prueba@example.org','telefono':None,'programa':None,'modalidad':None,'contacto_autorizado':False}),
 ('Me equivoqué: el correo correcto es final@example.com, no anterior@example.com.',{'nombre':None,'email':'final@example.com','telefono':None,'programa':None,'modalidad':None,'contacto_autorizado':None}),
 ('¿Puedo estudiar online? No deseo dejar mis datos.',{'nombre':None,'email':None,'telefono':None,'programa':None,'modalidad':'online','contacto_autorizado':False})]
 for i in range(25):
  inp,exp=examples[i%len(examples)]
  extract.append({'id':f'e-{i+1:03d}','task':'extract','input':inp,'expected':exp,'category':['dato inventado','corrección ignorada','negación ignorada','dato ausente','normalización incorrecta'][i%5],'difficulty':'difícil' if i%5 in [2,3] else 'fácil','note':'CASO SINTÉTICO / BORRADOR: revisar; duplicado de plantilla, sustituir por variación independiente.'})
# Solo preguntas con evidencia literal verificada son evaluables; se crean borradores de revisión.
question_templates=['¿Qué indica el documento acerca de {tema}?','¿Cuáles son las condiciones sobre {tema}?','¿Qué información oficial existe sobre {tema}?','¿Cómo se aplica lo señalado sobre {tema}?','¿Qué debería consultar un postulante respecto de {tema}?']
temas=['requisitos de admisión','aranceles','modalidad','postulación','matrícula','fechas','duración','convalidaciones','contacto','tratamiento de datos']
answers=[]
for i in range(25):
 candidate=CHUNKS[i%len(CHUNKS)] if CHUNKS else None
 # Preguntas sin evidencia: no se puede asumir que la respuesta está ausente sin revisar todo el corpus.
 answers.append({'id':f'q-{i+1:03d}','task':'answer','input':question_templates[i%5].format(tema=temas[i%len(temas)]),
 'expected':{'answer':'PENDIENTE_REVISION_HUMANA','evidence':([] if not candidate else [{'doc_id':candidate['doc_id'],'quote':candidate['text'][:min(180,len(candidate['text']))]}]),'answerable':None},
 'category':['fuente equivocada','cita sin respaldo','sin evidencia','versión desactualizada','multidocumento'][i%5],
 'difficulty':'difícil' if i%3==0 else 'fácil',
 'note':'BORRADOR NO ANOTADO: la evidencia sugerida NO garantiza responder a la pregunta. Revisar pregunta, cita literal, answerable y respuesta.'})
CASES=extract+answers
cases_path=ROOT/'eval/cases.jsonl'
cases_path.write_text(''.join(json.dumps(c,ensure_ascii=False)+'\n' for c in CASES),encoding='utf-8')
print('Casos:',len(CASES),'extracción:',sum(c['task']=='extract' for c in CASES),'respuesta:',sum(c['task']=='answer' for c in CASES))
print('ATENCIÓN: los 25 casos de respuesta requieren anotación manual y los de extracción revisión.')

Casos preliminares importados: 0
Casos: 50 extracción: 25 respuesta: 25
ATENCIÓN: los 25 casos de respuesta requieren anotación manual y los de extracción revisión.


## 4 · Anotación y validación humana (obligatoria)
En `eval/cases.jsonl`, para cada respuesta: redactar pregunta específica, `expected.answer` sustentada en documentos, `answerable` verdadero/falso, y una o varias citas **literales** en `expected.evidence`. Para una pregunta sin respaldo, usar `answerable=false`, `answer='sin evidencia'` y `evidence=[]` **solo tras revisar el corpus**. La herramienta detecta citas inexistentes y anotaciones pendientes; no puede juzgar por sí sola si una cita realmente responde a la pregunta.

In [16]:
rules='# Reglas de anotación — UCTVirtual Advance\n\n1. Fuente autorizada: solo documentos institucionales oficiales vigentes cargados en corpus/originales. No inferir requisitos, aranceles, fechas ni programas a partir de conocimiento general.\n2. `nombre`: nombre explícito del usuario; si es ambiguo, null. `email`: última corrección explícita, si es válido. `telefono`: dígitos normalizados, sin inventar prefijos. `programa`: nombre textual explícito, sin validarlo contra oferta si no hay ficha. `modalidad`: solo si es explícita. `contacto_autorizado`: true solo con consentimiento afirmativo explícito; false ante negativa; null si no se menciona.\n3. No inferir autorización de contacto por el hecho de proporcionar un correo o teléfono. No usar información real de estudiantes en el conjunto público.\n4. `answerable=true` únicamente cuando el corpus vigente contiene evidencia suficiente. `expected.evidence` registra `doc_id` y cita literal exacta; si se requieren dos fuentes, incluir ambas. `answerable=false` cuando, tras revisión, no existe respaldo; respuesta: "sin evidencia".\n5. En conflicto de versiones, priorizar documento vigente según fecha/aprobación comprobada; si no se puede establecer, abstenerse y derivar.\n6. Categorías de falla: dato inventado; corrección ignorada; negación ignorada; dato ausente; normalización incorrecta; fuente equivocada; cita sin respaldo; sin evidencia; versión desactualizada; multidocumento. Para el informe agrupar en 8 categorías si se requiere.\n7. Una persona revisa todos los casos, en especial los generados por IA o plantilla, y registra fecha y responsable. Los casos pendientes no son aptos para reportar métricas definitivas.\n'
(ROOT/'eval/annotation-rules.md').write_text(rules,encoding='utf-8')
def validar_casos(cases):
 errs=[]; texts={p.stem:p.read_text(encoding='utf-8') for p in (ROOT/'corpus/textos').glob('*.txt')}
 for c in cases:
  if c['task']=='answer':
   exp=c['expected']; cid=c['id']
   if not isinstance(exp.get('answerable'),bool) or exp.get('answer')=='PENDIENTE_REVISION_HUMANA':errs.append((cid,'anotación pendiente'))
   for e in exp.get('evidence',[]):
    if e.get('doc_id') not in texts or e.get('quote','') not in texts.get(e.get('doc_id'),''):errs.append((cid,'cita no literal o documento ausente'))
   if exp.get('answerable') is False and exp.get('evidence'):errs.append((cid,'sin evidencia con citas'))
 return errs
errors=validar_casos(CASES)
print('Problemas de anotación:',len(errors));display(pd.DataFrame(errors,columns=['caso','problema']).head(15))

Problemas de anotación: 25


,caso,problema
0,q-001,anotación pendiente
1,q-002,anotación pendiente
2,q-003,anotación pendiente
3,q-004,anotación pendiente
4,q-005,anotación pendiente
5,q-006,anotación pendiente
6,q-007,anotación pendiente
7,q-008,anotación pendiente
8,q-009,anotación pendiente
9,q-010,anotación pendiente


## 5 · Línea base reproducible: extracción por reglas y recuperación TF-IDF
La extracción se realiza con expresiones regulares, sin LLM. Para respuestas, la recuperación es TF-IDF y la generación es **extractiva**: devuelve un fragmento con cita si encuentra coincidencia, o «sin evidencia». Esta solución deliberadamente simple permite calcular métricas sin costo. **No se reportan métricas definitivas hasta que el conjunto esté revisado.**

In [17]:
def extraer_baseline(s):
 emails=re.findall(r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}',s)
 tel=re.findall(r'(?:\+?56[ -]?)?9[ -]?(?:\d[ -]?){8}',s)
 lower=s.lower()
 consent=False if re.search(r'no (?:quiero|autorizo|deseo).{0,35}(?:contact|llam|datos)',lower) else (True if re.search(r'autorizo.{0,30}contact',lower) else None)
 return {'nombre':None,'email':emails[-1] if emails else None,'telefono':re.sub(r'\D','',tel[-1]) if tel else None,'programa':None,'modalidad':'online' if re.search(r'\bonline\b',lower) else None,'contacto_autorizado':consent}
def responder_baseline(q,k=5,chunks=None):
 hits=buscar(q,k,chunks)
 if not hits:return {'answer':'sin evidencia','evidence':[],'answerable':False,'retrieved_chunk_ids':[],'retrieved_doc_ids':[]}
 h=hits[0]
 return {'answer':h['text'][:350],'evidence':[{'doc_id':h['doc_id'],'quote':h['text'][:min(180,len(h['text']))]}],'answerable':True,'retrieved_chunk_ids':[x['chunk_id'] for x in hits],'retrieved_doc_ids':[x['doc_id'] for x in hits]}
def correr(cases,k=5,chunks=None):
 out=[]
 for c in cases:
  t0=time.perf_counter()
  pred=extraer_baseline(c['input']) if c['task']=='extract' else responder_baseline(c['input'],k,chunks)
  out.append({'id':c['id'],'task':c['task'],'prediction':pred,'latency_ms':round((time.perf_counter()-t0)*1000,3)})
 return out
# Recargar el archivo tras la edición manual en Colab (o sustituirlo mediante carga)
CASES=[json.loads(s) for s in cases_path.read_text(encoding='utf-8').splitlines() if s.strip()]
OUTPUTS=correr(CASES)
outpath=ROOT/'eval/baseline/outputs.jsonl'
outpath.write_text(''.join(json.dumps(o,ensure_ascii=False)+'\n' for o in OUTPUTS),encoding='utf-8')
print('Salidas guardadas:',len(OUTPUTS))

Salidas guardadas: 50


## 6 · Métricas recalculables, sin llamadas al modelo
Extracción: exactitud por campo, exactitud del registro completo, falsos positivos en campos esperados null y consentimiento. Respuestas: recall@5 por documento y por cita literal, validez literal de citas, abstención en preguntas no respondibles. Se reportan denominadores. Los casos sin anotación se excluyen y se informa cuántos faltan.

In [18]:
EVAL_SCRIPT=r'''import json, sys
from pathlib import Path

def load(p):return [json.loads(s) for s in Path(p).read_text(encoding='utf-8').splitlines() if s.strip()]
def evaluate(cases,outputs,corpus_dir):
 pred={o['id']:o['prediction'] for o in outputs}; texts={p.stem:p.read_text(encoding='utf-8') for p in Path(corpus_dir).glob('*.txt')}
 counts={}; details=[]
 def inc(k,ok):
  counts.setdefault(k,[0,0]);counts[k][1]+=1;counts[k][0]+=int(bool(ok))
 for c in cases:
  p=pred.get(c['id']); e=c['expected']
  if p is None:continue
  if c['task']=='extract':
   inc('extract_registro_completo',all(p.get(k)==v for k,v in e.items()))
   for k,v in e.items():
    inc('extract_'+k,p.get(k)==v)
    if v is None:inc('extract_sin_inventar_'+k,p.get(k) is None)
  else:
   if not isinstance(e.get('answerable'),bool) or e.get('answer')=='PENDIENTE_REVISION_HUMANA':continue
   if e['answerable']:
    expected=e.get('evidence',[]);retrieved=set(p.get('retrieved_doc_ids',[]))
    inc('answer_recall_doc_at5',bool(expected) and all(x['doc_id'] in retrieved for x in expected))
    # Cita recuperada: al menos un chunk de cada documento contiene la frase literal esperada.
    retrieved_chunks=[chunk_map.get(i,'') for i in p.get('retrieved_chunk_ids',[])]
    inc('answer_recall_cita_at5',bool(expected) and all(any(x['quote'] in t for t in retrieved_chunks) for x in expected))
   else:inc('answer_abstencion',p.get('answerable') is False and p.get('answer')=='sin evidencia')
   for cite in p.get('evidence',[]):inc('answer_cita_literal',bool(cite.get('quote')) and cite['quote'] in texts.get(cite.get('doc_id'),''))
 return {k:{'aciertos':v[0],'total':v[1],'valor':round(v[0]/v[1],4) if v[1] else None} for k,v in counts.items()}
if __name__=='__main__':
 base=Path(sys.argv[1]); cs=load(base/'eval/cases.jsonl'); outs=load(base/'eval/baseline/outputs.jsonl')
 chunk_map={c['chunk_id']:c['text'] for c in load(base/'corpus/chunks.jsonl')}
 result=evaluate(cs,outs,base/'corpus/textos'); print(json.dumps(result,ensure_ascii=False,indent=2))
 (base/'eval/baseline/metrics.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
'''
(ROOT/'eval/baseline/evaluate.py').write_text(EVAL_SCRIPT,encoding='utf-8')
import subprocess,sys
p=subprocess.run([sys.executable,str(ROOT/'eval/baseline/evaluate.py'),str(ROOT)],capture_output=True,text=True)
print(p.stdout if p.returncode==0 else p.stderr)
print('Casos pendientes:',len(validar_casos(CASES)))

{
  "extract_registro_completo": {
    "aciertos": 10,
    "total": 25,
    "valor": 0.4
  },
  "extract_nombre": {
    "aciertos": 15,
    "total": 25,
    "valor": 0.6
  },
  "extract_email": {
    "aciertos": 20,
    "total": 25,
    "valor": 0.8
  },
  "extract_telefono": {
    "aciertos": 25,
    "total": 25,
    "valor": 1.0
  },
  "extract_sin_inventar_telefono": {
    "aciertos": 20,
    "total": 20,
    "valor": 1.0
  },
  "extract_programa": {
    "aciertos": 25,
    "total": 25,
    "valor": 1.0
  },
  "extract_sin_inventar_programa": {
    "aciertos": 25,
    "total": 25,
    "valor": 1.0
  },
  "extract_modalidad": {
    "aciertos": 25,
    "total": 25,
    "valor": 1.0
  },
  "extract_sin_inventar_modalidad": {
    "aciertos": 20,
    "total": 20,
    "valor": 1.0
  },
  "extract_contacto_autorizado": {
    "aciertos": 25,
    "total": 25,
    "valor": 1.0
  },
  "extract_sin_inventar_email": {
    "aciertos": 10,
    "total": 10,
    "valor": 1.0
  },
  "extract_sin_inve

## 7 · Comparar alternativas A/B (desempeño, costo, latencia, privacidad, riesgo)
**A:** TF-IDF con fragmentos de 900 caracteres, k=5. **B:** TF-IDF con fragmentos de 450 caracteres, k=5. Ambas variantes son medibles y gratuitas; para evaluar generación con LLM se debe ejecutar y guardar una tercera alternativa opcional. El costo de llamadas LLM no se estima sin proveedor, precios y tokens medidos.

In [19]:
A=CHUNKS;B=construir_chunks(max_chars=450,overlap=70)
rows=[]
for name,ch in [('A - 900 caracteres',A),('B - 450 caracteres',B)]:
 outs=correr(CASES,chunks=ch)
 valid=[c for c in CASES if c['task']=='answer' and isinstance(c['expected'].get('answerable'),bool)]
 ok=0;total=0
 for c in valid:
  if c['expected']['answerable']:
   total+=1;hits=buscar(c['input'],5,ch)
   ok+=int(bool(c['expected'].get('evidence')) and all(any(e['quote'] in h['text'] for h in hits) for e in c['expected']['evidence']))
 rows.append({'alternativa':name,'fragmentos':len(ch),'recall_cita@5':(ok/total if total else None),'casos_respondibles_anotados':total,'latencia_mediana_ms':round(pd.Series([o['latency_ms'] for o in outs if o['task']=='answer']).median(),3),'costo_API_USD':0,'privacidad':'local: texto no enviado a proveedor','riesgo':'TF-IDF falla con paráfrasis; no valida vigencia ni pertinencia semántica'})
display(pd.DataFrame(rows))
print('Sin anotación revisada, recall_cita@5 permanece sin resultado (None).')

,alternativa,fragmentos,recall_cita@5,casos_respondibles_anotados,latencia_mediana_ms,costo_API_USD,privacidad,riesgo
0,A - 900 caracteres,0,None,0,0.001,0,local: texto no enviado a proveedor,TF-IDF falla con paráfrasis; no valida vigenci...
1,B - 450 caracteres,0,None,0,0.002,0,local: texto no enviado a proveedor,TF-IDF falla con paráfrasis; no valida vigenci...


Sin anotación revisada, recall_cita@5 permanece sin resultado (None).


## 8 · Documento Hito 1 y protocolo
El informe incorpora métricas medidas **solo si existen**, umbrales propuestos y la condición de no entrega si faltan anotaciones. Los umbrales se justifican por el costo de orientar erróneamente al postulante. Revisar y personalizar antes de enviar.

In [20]:
metrics_path=ROOT/'eval/baseline/metrics.json'
metrics=json.loads(metrics_path.read_text(encoding='utf-8')) if metrics_path.exists() else {}
pending=len(validar_casos(CASES))
metric_lines='\n'.join(f"- {k}: {v['aciertos']}/{v['total']} ({v['valor']})" for k,v in metrics.items())
report=f"""# Hito 1 · UCTVirtual AI Advisor

## 1. Problema, usuarios y criterios de éxito
Usuarios: interesados en programas Advance, consejeros de admisión y responsables de admisión UCTVirtual. El asistente recibe conversaciones de texto y documentos institucionales autorizados; entrega datos estructurados y respuestas con fuentes. Un error puede inducir decisiones equivocadas de postulación, generar expectativas de aranceles o requisitos incorrectos o comprometer datos personales. Error inaceptable: inventar requisitos, aranceles, plazos o consentimiento.

**Criterios de extracción (metas propuestas):** exactitud por campo >= 95%; consentimiento >= 100%; falsos positivos en datos ausentes = 0. La autorización de contacto requiere tolerancia cero por privacidad. **Criterios de respuestas:** recall de cita@5 >= 90%; citas literales y pertinentes = 100%; abstención cuando falta respaldo >= 95%. Se requiere revisión humana de pertinencia: la coincidencia literal no basta.

**Impacto comercial (fuera del alcance medido en Hito 1):** tiempo de primera respuesta, porcentaje de leads calificados, derivaciones efectivas, conversión lead-matrícula. Comparar contra línea base histórica solo cuando se homologuen períodos y definiciones.

## 2. Conjunto de evaluación
Archivo `eval/cases.jsonl`: {len(CASES)} casos ({sum(c['task']=='extract' for c in CASES)} de extracción, {sum(c['task']=='answer' for c in CASES)} de respuesta). Reglas en `eval/annotation-rules.md`. Pendientes detectados: {pending}. **No declarar los casos como validados mientras existan pendientes.** Los documentos y citas deben ser oficiales y literales. Registrar responsable de revisión y fecha.

## 3. Línea base
Extracción determinista mediante expresiones regulares; recuperación TF-IDF; respuesta extractiva. Salidas persistidas en `eval/baseline/outputs.jsonl`, script `eval/baseline/evaluate.py`, métricas en `eval/baseline/metrics.json`.

{metric_lines}

**Limitación:** métricas de respuesta calculadas solo sobre anotaciones terminadas. Ninguna cifra de respuesta constituye validación si hay casos pendientes. Categorizar errores tras la revisión humana y agregar tabla de recuentos.

## 4. Alternativas comparadas
A: TF-IDF, fragmentos de 900 caracteres, k=5. B: TF-IDF, fragmentos de 450 caracteres, k=5. Ambas operan localmente sin llamadas API y protegen la privacidad al no transmitir contenido. Comparación ejecutable en celda 7. El tamaño menor puede mejorar precisión local pero perder contexto; el mayor puede mejorar continuidad y añadir ruido. La latencia depende del entorno Colab. **Recomendación provisional:** mantener A como baseline y elegir por recall de citas medido después de anotar. Cambiar si B logra mejor recall sin degradación significativa de latencia. Para comparar con un LLM, medir tokens, costo publicado, latencia, seguridad y citas; no inventar cifras.

## 5. Uso de IA declarado
ChatGPT se utilizó para estructurar el notebook, proponer reglas y casos sintéticos de evaluación y redactar borradores. **No se ha realizado revisión humana de las anotaciones en esta ejecución.** El equipo debe declarar los asistentes efectivamente usados, la revisión humana realizada y los controles ejecutados antes de entregar.

## Verificaciones pendientes antes de publicar
- [ ] Documentos oficiales autorizados cargados y vigencia comprobada.
- [ ] 50 casos únicos y anotados revisados por integrantes del equipo.
- [ ] Preguntas con evidencia literal y respuesta respaldada.
- [ ] Errores de anotación = 0; citas pertinentes verificadas manualmente.
- [ ] Línea base ejecutada y recalculable desde salidas guardadas.
- [ ] Recuento de fallas por categoría.
- [ ] Alternativas comparadas con métricas y riesgos documentados.
- [ ] Commit y etiqueta `hito1` creados en GitHub; declaración de IA en Canvas.
"""
(ROOT/'docs/hito1.md').write_text(report,encoding='utf-8')
print('Informe generado:',ROOT/'docs/hito1.md')

Informe generado: /content/uctvirtual_hito1/docs/hito1.md


## 9 · Exportar ZIP para GitHub / Canvas
El ZIP contendrá el corpus original y el texto extraído. **No incluir documentos internos/confidenciales ni datos personales sin autorización de la UCT.** Puedes excluir `corpus/originales` antes de publicarlo; para reproducibilidad, adjuntar versiones autorizadas o referencias y hashes.

In [21]:
# Por defecto NO exporta documentos originales: evitar divulgación accidental.
INCLUIR_ORIGINALES=False
zip_out=ROOT.parent/'UCTVirtual_Hito1_Colab_Entrega.zip'
with zipfile.ZipFile(zip_out,'w',zipfile.ZIP_DEFLATED) as z:
 for p in ROOT.rglob('*'):
  if not p.is_file() or p==SEED_ZIP:continue
  rel=p.relative_to(ROOT)
  if not INCLUIR_ORIGINALES and rel.parts[:2]==('corpus','originales'):continue
  z.write(p,rel.as_posix())
print('ZIP:',zip_out,'tamaño KB:',round(zip_out.stat().st_size/1024,1))
try:
 from google.colab import files
 files.download(str(zip_out))
except ImportError:pass
print('IMPORTANTE: no subir si hay anotaciones pendientes:',len(validar_casos(CASES)))

ZIP: /content/UCTVirtual_Hito1_Colab_Entrega.zip tamaño KB: 6.6


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

IMPORTANTE: no subir si hay anotaciones pendientes: 25


### Checklist final
1. Cargar fuentes oficiales, revisar fechas de vigencia y volver a ejecutar.
2. Editar 50 casos en `eval/cases.jsonl` (sin datos personales reales); asegurar 25 preguntas respaldadas o correctamente abstencionistas y 25 casos de extracción diversos.
3. Volver a ejecutar desde **celda 5** después de editar; comprobar 0 pendientes en la celda 4.
4. Revisar manualmente si las citas son suficientes, no solo literales; documentar errores por categoría.
5. Descargar ZIP, subir archivos al repositorio del grupo, crear etiqueta `hito1` y declarar el uso de IA en Canvas.

**Nota:** el notebook es una plataforma de evaluación; no constituye aún un chatbot productivo, CRM ni integración real con WhatsApp.